# Data Prep v2 — 端到端数据准备管线（方案二：CWE-MIXED）

**方案二**全流程整合（相对方案一的关键差异：投毒目标按 **CWE-MIXED 单桶** 统一处理）：
1. 语料 & 查询集构建（Vul + BFP + white/gray/black query_set）
2. 三个向量库构建（历史查询库 query.yml / BFP-APR / Vul-Vinj）
3. 关联 CVE 实例与历史查询：
   - 3.1 BFP↔CVE 匹配（以 CVE 的 vul_code 检索历史查询库）
   - 3.2 含安全隐患的历史查询集 `unsafe_query_set_v2.json`（**跨 CWE_LIST 汇总去重统一归入 `CWE-MIXED`** + 关联 CVE 数 ≥ τ 过滤）
   - 3.3 测试集划分 `test_query_set_v2.json`（unsafe 中按 8:2 切出 20%）
4. 投毒目标选择（候选池 = unsafe − test，对 **CWE-MIXED 单桶** 聚类 + 投毒预算分配 + 簇内随机采样）
5. 正样本选择（Vinj 库检索相关 CVE + 加权采样 BFP）

> **使用方法**：按顺序运行所有 cell；每个写盘/建库步骤均有幂等检查，已完成则自动跳过。

## Section 0 — 全局初始化

In [1]:
# ========== Section 0: 全局初始化 ==========
import os
import sys
import gc
import json
import random
import torch
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

# ── 项目根目录上溯 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

from src.utils.io import load_json, load_jsonl, load_json_stream, save_json, load_yaml
from src.utils.log import setup_logging
from src.models.retriever.Base import create_embedder
from src.rag.bfp.milvus_client import BFPMilvusClient
from src.rag.bfp.retriever import BFPRetriever
from src.rag.vul.milvus_client import VulMilvusClient
from src.rag.vul.retriever import VulRetriever

setup_logging()


# ── 嵌入模型按需加载/释放的辅助函数 ──
def load_embedder_from_db_config(db_config: dict):
    """从数据库配置的 database.model 字段加载嵌入模型。"""
    os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
    cfg_path = PROJECT_ROOT / db_config["database"]["model"]
    return create_embedder(load_yaml(str(cfg_path)))

def release_embedder(emb) -> None:
    """释放嵌入模型并清空 GPU 显存。"""
    del emb
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("[GPU] 嵌入模型已释放")

PROJECT_ROOT: <PROJECT_ROOT>


## Section 1 — 语料 & 查询集构建

### Section 1.1 — Vul 语料构建

In [2]:
# ========== Section 1.1: Vul 语料构建 ==========
# 产出：data/corpus/vul/clustered_rag_data.jsonl

VUL_CORPUS_OUT = PROJECT_ROOT / "data/corpus/vul/clustered_rag_data.jsonl"

if VUL_CORPUS_OUT.exists():
    print(f"[SKIP] Vul 语料已存在：{VUL_CORPUS_OUT}")
    df_vul = pd.read_json(str(VUL_CORPUS_OUT), lines=True)
    print(f"       加载 {len(df_vul)} 条")
else:
    # ── 1. 加载原始数据集（已由 vul/dataset_formatting.ipynb 预先格式化）──
    bigvul_df   = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/BigVul.jsonl"),   lines=True)
    cvefixes_df = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/CVEfixes.jsonl"), lines=True)
    reposvul_df = pd.read_json(str(PROJECT_ROOT / "data/raw/vul/ReposVul_v2.jsonl"), lines=True)
    df_vul = pd.concat([bigvul_df, cvefixes_df, reposvul_df], ignore_index=True)

    print(f"原始合并: {len(df_vul)} 条")

    # ── 2. 按 vul_code + cwe_id 去重（同一漏洞代码可能对应不同 CWE，需保留）──
    before = len(df_vul)
    df_vul = df_vul.drop_duplicates(subset=["vul_code", "cwe_id"], keep="first").reset_index(drop=True)
    print(f"去重: {before} → {len(df_vul)} (移除 {before - len(df_vul)})")

    # ── 3. 嵌入 + KMeans 聚类 ──
    all_codes = df_vul["vul_code"].astype(str).tolist()
    vul_embedder = load_embedder_from_db_config(
        {"database": {"model": "configs/models/harrier-oss-v1-0.6b.yml"}}
    )
    encoded = np.asarray(vul_embedder.embed_documents(all_codes))
    release_embedder(vul_embedder)  # KMeans 不需要 GPU，提前释放显存
    print(f"嵌入维度: {encoded.shape}")

    kmeans = KMeans(n_clusters=5, random_state=42)
    kmeans.fit(encoded)
    df_vul["cluster_id"] = kmeans.labels_
    print(f"KMeans 聚类完成，各簇分布:\n{df_vul['cluster_id'].value_counts().sort_index()}")

    # ── 4. 写出 ──
    VUL_CORPUS_OUT.parent.mkdir(parents=True, exist_ok=True)
    df_vul.to_json(str(VUL_CORPUS_OUT), orient="records", lines=True)
    print(f"✅ 已写入 {len(df_vul)} 条 → {VUL_CORPUS_OUT}")

df_vul.head()

[SKIP] Vul 语料已存在：<PROJECT_ROOT>/data/corpus/vul/clustered_rag_data.jsonl
       加载 17576 条


,id,source,language,cwe_id,cve_id,cwe_desc,cve_desc,vul_code,fixed_code,diff,cluster_id
0,0,BigVul,C,CWE-264,CVE-2015-8467,"- CWE ID: CWE-264\n- Name: Permissions, Privil...",The samldb_check_user_account_control_acl func...,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...,"--- vul_code\n+++ fixed_code\n@@ -9,12 +9,15 @...",2
1,1,BigVul,C,CWE-119,CVE-2015-8382,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The match function in pcre_exec.c in PCRE befo...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,"--- vul_code\n+++ fixed_code\n@@ -87,6 +87,7 @...",3
2,2,BigVul,C,CWE-119,CVE-2013-6712,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The scan function in ext/date/lib/parse_iso_in...,static int scan(Scanner *s) {\n uchar *cursor...,static int scan(Scanner *s) {\n uchar *cursor...,"--- vul_code\n+++ fixed_code\n@@ -147,7 +147,7...",3
3,3,BigVul,C,CWE-310,CVE-2013-6449,- CWE ID: CWE-310\n- Name: Cryptographic Issue...,The ssl_get_algorithm2 function in ssl/s3_lib....,long ssl_get_algorithm2(SSL *s) {\n long alg2...,long ssl_get_algorithm2(SSL *s) {\n long alg2...,"--- vul_code\n+++ fixed_code\n@@ -1,6 +1,6 @@\...",4
4,4,BigVul,C,CWE-119,CVE-2013-6420,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The asn1_time_to_time_t function in ext/openss...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,"--- vul_code\n+++ fixed_code\n@@ -13,20 +13,30...",3


### Section 1.2 — BFP 语料构建

In [3]:
# ========== Section 1.2: BFP 语料构建 ==========
# 产出：data/corpus/bfp/rag_data.jsonl

CORPUS_SIZE = 50000
SEED = 42
BFP_CORPUS_OUT = PROJECT_ROOT / "data/corpus/bfp/rag_data.jsonl"

# ── 工具函数 ──
def extract_fields(record: dict, source: str) -> dict:
    return {
        "language": record["language"],
        "buggy_code": record["buggy_code"],
        "fixed_code": record["fixed_code"],
        "source": source,
    }

def assign_ids(data: list) -> list:
    return [{"id": i} | item for i, item in enumerate(data, 1)]

def write_jsonl(path, data: list):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"  ✅ 已写入 {len(data)} 条 → {path}")

if BFP_CORPUS_OUT.exists():
    print(f"[SKIP] BFP 语料已存在：{BFP_CORPUS_OUT}")
    corpus_dataset = load_jsonl(str(BFP_CORPUS_OUT))
    print(f"       加载 {len(corpus_dataset)} 条")
else:
    random.seed(SEED)

    # ── 读取原始数据（已由 bfp/dataset_formatting.ipynb 预先格式化）──
    coconut_data = []
    with open(PROJECT_ROOT / "data/raw/bfp/CoCoNut.jsonl", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                coconut_data.append(json.loads(line))
    with open(PROJECT_ROOT / "data/raw/bfp/Codeflaws.json", encoding="utf-8") as f:
        codeflaws_data = json.load(f)
    with open(PROJECT_ROOT / "data/raw/bfp/Deepfix.json", encoding="utf-8") as f:
        deepfix_data = json.load(f)

    random.shuffle(coconut_data)
    random.shuffle(codeflaws_data)
    random.shuffle(deepfix_data)

    print(f"原始数据: CoCoNut={len(coconut_data)}, Codeflaws={len(codeflaws_data)}, DeepFix={len(deepfix_data)}")

    # ── 按占比分配语料库样本数 ──
    TOTAL_RAW = len(coconut_data) + len(codeflaws_data) + len(deepfix_data)
    CORPUS_COCONUT   = round(CORPUS_SIZE * len(coconut_data)   / TOTAL_RAW)
    CORPUS_CODEFLAWS = round(CORPUS_SIZE * len(codeflaws_data) / TOTAL_RAW)
    CORPUS_DEEPFIX   = CORPUS_SIZE - CORPUS_COCONUT - CORPUS_CODEFLAWS

    print(f"语料库分配: CoCoNut={CORPUS_COCONUT}, Codeflaws={CORPUS_CODEFLAWS}, DeepFix={CORPUS_DEEPFIX}")

    corpus_coconut   = coconut_data[:CORPUS_COCONUT]
    corpus_codeflaws = codeflaws_data[:CORPUS_CODEFLAWS]
    corpus_deepfix   = deepfix_data[:CORPUS_DEEPFIX]

    corpus_dataset = assign_ids(
        [extract_fields(r, "CoCoNut")   for r in corpus_coconut] +
        [extract_fields(r, "Codeflaws") for r in corpus_codeflaws] +
        [extract_fields(r, "DeepFix")   for r in corpus_deepfix]
    )
    assert len(corpus_dataset) == CORPUS_SIZE

    write_jsonl(BFP_CORPUS_OUT, corpus_dataset)

print(f"BFP 语料库规模: {len(corpus_dataset)} 条")

[SKIP] BFP 语料已存在：<PROJECT_ROOT>/data/corpus/bfp/rag_data.jsonl


2026-06-30 01:25:07 [INFO] src.utils.io: Loaded 50000 records from <PROJECT_ROOT>/data/corpus/bfp/rag_data.jsonl


       加载 50000 条
BFP 语料库规模: 50000 条


### Section 1.3 — BFP 查询集构建（white / gray / black）

In [4]:
# ========== Section 1.3: BFP 查询集构建 ==========
# 产出：
#   data/query/white/query_set.jsonl  （完全 ⊆ 语料库）
#   data/query/gray/query_set.jsonl   （OVERLAP_RATE 来自语料库 + 余下来自语料库外）
#   data/query/black/query_set.jsonl  （全部来自语料库外）

# QUERY_SIZE    = 25000
QUERY_SIZE    = 50000
OVERLAP_RATE  = 0.1
SEED          = 42

WHITE_QUERY_OUT = PROJECT_ROOT / "data/query/white/query_set.jsonl"
GRAY_QUERY_OUT  = PROJECT_ROOT / "data/query/gray/query_set.jsonl"
BLACK_QUERY_OUT = PROJECT_ROOT / "data/query/black/query_set.jsonl"

all_done = WHITE_QUERY_OUT.exists() and GRAY_QUERY_OUT.exists() and BLACK_QUERY_OUT.exists()

if all_done:
    print("[SKIP] 三套查询集已存在")
    white_dataset = load_jsonl(str(WHITE_QUERY_OUT))
    gray_dataset  = load_jsonl(str(GRAY_QUERY_OUT))
    black_dataset = load_jsonl(str(BLACK_QUERY_OUT))
    print(f"       white={len(white_dataset)}, gray={len(gray_dataset)}, black={len(black_dataset)}")
else:
    rng = random.Random(SEED)

    # ── corpus_pool：直接使用已生成的语料库 ──
    corpus_pool = corpus_dataset[:]  # 来自 Section 1.2，已格式化

    # ── outside_pool：原始数据集 − 语料库 ──
    # 用 buggy_code 做精确去重（语料库内条目均来自原始数据集前 N 条）
    corpus_buggy_set = {item["buggy_code"] for item in corpus_pool}

    def _load_raw_bfp():
        coc = []
        with open(PROJECT_ROOT / "data/raw/bfp/CoCoNut.jsonl", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    coc.append(json.loads(line))
        with open(PROJECT_ROOT / "data/raw/bfp/Codeflaws.json", encoding="utf-8") as f:
            clf = json.load(f)
        with open(PROJECT_ROOT / "data/raw/bfp/Deepfix.json", encoding="utf-8") as f:
            dpf = json.load(f)
        return coc, clf, dpf

    def infer_source(record: dict) -> str:
        if "user_id" in record:
            return "DeepFix"
        if "bug_type" in record:
            return "Codeflaws"
        return "CoCoNut"

    try:
        # 如果 Section 1.2 的变量还在内存中直接利用
        _raw_all = (
            [extract_fields(r, "CoCoNut")   for r in coconut_data] +
            [extract_fields(r, "Codeflaws") for r in codeflaws_data] +
            [extract_fields(r, "DeepFix")   for r in deepfix_data]
        )
    except NameError:
        coc, clf, dpf = _load_raw_bfp()
        _raw_all = (
            [extract_fields(r, "CoCoNut")   for r in coc] +
            [extract_fields(r, "Codeflaws") for r in clf] +
            [extract_fields(r, "DeepFix")   for r in dpf]
        )

    outside_pool = [
        item for item in _raw_all
        if item["buggy_code"] not in corpus_buggy_set
    ]
    print(f"outside_pool 大小: {len(outside_pool)} 条（已由 bfp/dataset_formatting.ipynb 预先格式化）")

    # ── 抽取三套查询集 ──
    overlap_n = int(QUERY_SIZE * OVERLAP_RATE)   # gray 中来自语料库的部分
    outside_n = QUERY_SIZE - overlap_n           # gray 中来自语料库外的部分

    assert len(corpus_pool) >= QUERY_SIZE, "corpus_pool 不足以支撑 white 查询集"
    assert len(outside_pool) >= QUERY_SIZE + overlap_n, "outside_pool 数量不足"

    white_raw  = rng.sample(corpus_pool, QUERY_SIZE)
    gray_corp  = rng.sample(corpus_pool, overlap_n)
    gray_out   = rng.sample(outside_pool, outside_n)
    gray_raw   = gray_corp + gray_out
    black_raw  = rng.sample(outside_pool, QUERY_SIZE)

    white_dataset = assign_ids(white_raw)
    gray_dataset  = assign_ids(gray_raw)
    black_dataset = assign_ids(black_raw)

    write_jsonl(WHITE_QUERY_OUT, white_dataset)
    write_jsonl(GRAY_QUERY_OUT,  gray_dataset)
    write_jsonl(BLACK_QUERY_OUT, black_dataset)

print(f"\n查询集规模汇总:")
print(f"  white : {len(white_dataset)} 条 → {WHITE_QUERY_OUT}")
print(f"  gray  : {len(gray_dataset)} 条 → {GRAY_QUERY_OUT}")
print(f"  black : {len(black_dataset)} 条 → {BLACK_QUERY_OUT}")

[SKIP] 三套查询集已存在


2026-06-30 01:25:16 [INFO] src.utils.io: Loaded 25000 records from <PROJECT_ROOT>/data/query/white/query_set.jsonl
2026-06-30 01:25:24 [INFO] src.utils.io: Loaded 25000 records from <PROJECT_ROOT>/data/query/gray/query_set.jsonl
2026-06-30 01:25:32 [INFO] src.utils.io: Loaded 25000 records from <PROJECT_ROOT>/data/query/black/query_set.jsonl


       white=25000, gray=25000, black=25000

查询集规模汇总:
  white : 25000 条 → <PROJECT_ROOT>/data/query/white/query_set.jsonl
  gray  : 25000 条 → <PROJECT_ROOT>/data/query/gray/query_set.jsonl
  black : 25000 条 → <PROJECT_ROOT>/data/query/black/query_set.jsonl


## Section 2 — 向量库构建

### Section 2.1 — 历史查询向量库

配置文件：`configs/database/bfp/query.yml`  
以 white/gray/black 历史查询集（`query_set.jsonl`）为语料构建历史查询向量库，语料库和模型文件需要在配置文件中进行设置

In [5]:
# ========== Section 2.1: 构建历史查询向量库 ==========
query_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/query.yml"))

# 路径绝对化
if not os.path.isabs(query_db_config["database"]["uri"]):
    query_db_config["database"]["uri"] = str(PROJECT_ROOT / query_db_config["database"]["uri"])
if not os.path.isabs(query_db_config["data"]["corpus"]):
    query_db_config["data"]["corpus"] = str(PROJECT_ROOT / query_db_config["data"]["corpus"])
if not os.path.isabs(query_db_config["data"]["query"]):
    query_db_config["data"]["query"] = str(PROJECT_ROOT / query_db_config["data"]["query"])

query_milvus = BFPMilvusClient(query_db_config)
col_name = query_db_config["database"]["collection_name"]

already_built = (
    query_milvus.client.has_collection(col_name)
    and query_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = query_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] 历史查询向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(query_db_config["data"]["corpus"])
    print(f"构建历史查询向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(query_db_config)
    query_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ 历史查询向量库构建完成")

query_milvus.close()
print("历史查询向量库 Done")

2026-06-30 01:25:32 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-06-30 01:25:32 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/query/gray/harrier-oss-v1-0.6b.db
2026-06-30 01:25:32 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-06-30 01:25:44 [INFO] src.rag.bfp.milvus_client: Milvus client closed


[SKIP] 历史查询向量库已存在，共 25000 条
历史查询向量库 Done


### Section 2.2 — BFP-APR 向量库

配置文件：`configs/database/bfp/apr.yml`  
语料库和模型文件需要在配置文件中进行设置

In [6]:
# ========== Section 2.2: 构建 BFP-APR 向量库 ==========
# 模型和 URI 均从 apr.yml 的 database.model / database.uri 读取，无需手动覆写

apr_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/apr.yml"))

# 路径绝对化
if not os.path.isabs(apr_db_config["database"]["uri"]):
    apr_db_config["database"]["uri"] = str(PROJECT_ROOT / apr_db_config["database"]["uri"])
if not os.path.isabs(apr_db_config["data"]["corpus"]):
    apr_db_config["data"]["corpus"] = str(PROJECT_ROOT / apr_db_config["data"]["corpus"])

apr_milvus = BFPMilvusClient(apr_db_config)
col_name = apr_db_config["database"]["collection_name"]

already_built = (
    apr_milvus.client.has_collection(col_name)
    and apr_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = apr_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] BFP-APR 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(apr_db_config["data"]["corpus"])
    print(f"构建 BFP-APR 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(apr_db_config)
    apr_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ BFP-APR 向量库构建完成")

apr_milvus.close()
print("BFP-APR 向量库 Done")

2026-06-30 01:25:44 [INFO] src.rag.bfp.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/gte-modernbert-base.yml 读取 dim=768
2026-06-30 01:25:44 [INFO] src.rag.bfp.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/bfp/apr/gte-modernbert-base.db
2026-06-30 01:25:45 [INFO] src.rag.bfp.milvus_client: Milvus client connected
2026-06-30 01:26:06 [INFO] src.rag.bfp.milvus_client: Milvus client closed


[SKIP] BFP-APR 向量库已存在，共 51655 条
BFP-APR 向量库 Done


### Section 2.3 — Vul-Vinj 向量库

配置文件：`configs/database/vul/vinj.yml`  
语料库和模型文件需要在配置文件中进行设置

In [7]:
# ========== Section 2.3: 构建 Vul-Vinj 向量库 ==========

vinj_db_config = load_yaml(str(PROJECT_ROOT / "configs/database/vul/vinj.yml"))

# 路径绝对化
if not os.path.isabs(vinj_db_config["database"]["uri"]):
    vinj_db_config["database"]["uri"] = str(PROJECT_ROOT / vinj_db_config["database"]["uri"])
if not os.path.isabs(vinj_db_config["data"]["corpus"]):
    vinj_db_config["data"]["corpus"] = str(PROJECT_ROOT / vinj_db_config["data"]["corpus"])

vinj_milvus = VulMilvusClient(vinj_db_config)
col_name = vinj_db_config["database"]["collection_name"]

already_built = (
    vinj_milvus.client.has_collection(col_name)
    and vinj_milvus.get_collection_stats()["row_count"] > 0
)

if already_built:
    rows = vinj_milvus.get_collection_stats()["row_count"]
    print(f"[SKIP] Vul-Vinj 向量库已存在，共 {rows} 条")
else:
    kb_data = load_jsonl(vinj_db_config["data"]["corpus"])
    print(f"构建 Vul-Vinj 向量库，数据量: {len(kb_data)} 条...")
    local_embedder = load_embedder_from_db_config(vinj_db_config)
    vinj_milvus.build_knowledge_base(local_embedder, kb_data)
    release_embedder(local_embedder)
    print(f"✅ Vul-Vinj 向量库构建完成")

vinj_milvus.close()
print("Vul-Vinj 向量库 Done")

2026-06-30 01:26:06 [INFO] src.rag.vul.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-06-30 01:26:06 [INFO] src.rag.vul.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/vul/harrier-oss-v1-0.6b.db
2026-06-30 01:26:07 [INFO] src.rag.vul.milvus_client: Milvus client connected
2026-06-30 01:26:13 [INFO] src.rag.vul.milvus_client: Milvus client closed


[SKIP] Vul-Vinj 向量库已存在，共 17576 条
Vul-Vinj 向量库 Done


## Section 3 — 代理查询集 & 测试查询集构建

### Section 3.1 — BFP 查询集 ↔ CVE 实例匹配

以 Vul 语料库中每条 CVE 实例的 `vul_code` 作为查询，从历史查询向量库（`query.yml`）检索相关 BFP，并将结果保存为检索结果文件。

In [8]:
# ========== Section 3.1: BFP 查询集 ↔ CVE 实例匹配 ==========
# 使用 query.yml 配置（data.corpus=历史查询集, data.query=CVE实例, query_target=vul_code）

query_db_config_s3 = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/query.yml"))

# 路径绝对化
if not os.path.isabs(query_db_config_s3["database"]["uri"]):
    query_db_config_s3["database"]["uri"] = str(PROJECT_ROOT / query_db_config_s3["database"]["uri"])
if not os.path.isabs(query_db_config_s3["data"]["corpus"]):
    query_db_config_s3["data"]["corpus"] = str(PROJECT_ROOT / query_db_config_s3["data"]["corpus"])
if not os.path.isabs(query_db_config_s3["data"]["query"]):
    query_db_config_s3["data"]["query"] = str(PROJECT_ROOT / query_db_config_s3["data"]["query"])

# 推导输出文件路径
top_k_s3              = query_db_config_s3["retrieval"].get("top_k", 10)
query_target_s3       = query_db_config_s3["retrieval"].get("query_target", "vul_code")
retrieval_strategy_s3 = query_db_config_s3["retrieval"].get("retrieval_strategy", "dense")
alpha_str_s3          = "1" if retrieval_strategy_s3 == "dense" else (
                        "0" if retrieval_strategy_s3 == "sparse" else
                        str(query_db_config_s3["retrieval"].get("alpha_weight", 0.5)))

# 加载查询数据（CVE 实例）
query_data_s3 = load_jsonl(query_db_config_s3["data"]["query"])
print(f"CVE 实例查询集: {len(query_data_s3)} 条")

# 加载 embedder 以获取模型名（用于构造结果文件名）
query_embedder_s3  = load_embedder_from_db_config(query_db_config_s3)
model_short_name   = query_embedder_s3.get_model_name(with_provider=False)

retrieval_results_filename = (
    f"num_{len(query_data_s3)}_alpha_{alpha_str_s3}"
    f"_target_{query_target_s3}_retriever_{model_short_name}_retrieval_results.json"
)
# PROXY_RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data/query/white" / retrieval_results_filename
# PROXY_RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data/query/gray" / retrieval_results_filename
PROXY_RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data/query/black" / retrieval_results_filename


print(f"检索结果将保存至: {PROXY_RETRIEVAL_RESULTS_PATH}")

if PROXY_RETRIEVAL_RESULTS_PATH.exists():
    release_embedder(query_embedder_s3)
    print(f"[SKIP] 检索结果文件已存在")
    proxy_retrieval_results = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))
    print(f"       加载 {len(proxy_retrieval_results)} 条")
else:
    # 打开历史查询向量库并执行检索
    query_milvus_s3 = BFPMilvusClient(query_db_config_s3)
    retriever_s3    = BFPRetriever(query_embedder_s3, query_milvus_s3)

    print(f"执行密集检索: top_k={top_k_s3}, query_target={query_target_s3}")
    proxy_retrieval_results = retriever_s3.dense_retrieve(
        query_data_s3, top_k=top_k_s3, query_target=query_target_s3
    )

    PROXY_RETRIEVAL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_json(proxy_retrieval_results, str(PROXY_RETRIEVAL_RESULTS_PATH))
    print(f"✅ 检索结果已保存: {len(proxy_retrieval_results)} 条 → {PROXY_RETRIEVAL_RESULTS_PATH}")

    query_milvus_s3.close()
    release_embedder(query_embedder_s3)

print("Section 3.1 Done")

2026-06-30 01:26:21 [INFO] src.utils.io: Loaded 17576 records from <PROJECT_ROOT>/data/corpus/vul/clustered_rag_data.jsonl


CVE 实例查询集: 17576 条


2026-06-30 01:26:21 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-06-30 01:26:21 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-06-30 01:26:21 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-06-30 01:26:21 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-06-30 01:26:21 [INFO] src.utils.device: 预筛选后的设备池: ['cuda:0']
2026-06-30 01:26:22 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8187 MiB)
<CONDA_ENV>/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-30 01:26:25 [INFO] src.models.retriever.Harrier: Loading model: microsoft/harrier-oss-v1-0.6b on cuda:0
2026-06-30 01:26:25 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: microsoft/harrier-oss-v1-0.6b
2026-06

检索结果将保存至: <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
[GPU] 嵌入模型已释放
[SKIP] 检索结果文件已存在


2026-06-30 01:27:10 [INFO] src.utils.io: Stream-loaded 17576 records from <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json


       加载 17576 条
Section 3.1 Done


### Section 3.2 v2 — 关联 CVE 实例与历史查询（含安全隐患的历史查询集）

遍历 BFP↔CVE 检索结果，**跨 CWE_LIST 中所有 CWE 类型汇总命中的 BFP**：仅保留以 `score >= RELEVANCE_THRESHOLD`、`rank <= TOP_K` 被 `CWE_LIST` 内 CWE 类型 CVE 实例召回的 BFP；统计每个 BFP 关联到的不同 CVE 数量，过滤掉关联数 `< CVE_ASSOCIATION_THRESHOLD` 的 BFP，汇总去重后**统一归入单键 `CWE-MIXED`**，存储为 `unsafe_query_set_v2.json`。

In [9]:
# ========== Section 3.2 v2: 关联 CVE 实例与历史查询（unsafe_query_set_v2，CWE-MIXED） ==========

CWE_LIST                  = ["CWE-125", "CWE-20", "CWE-119", "CWE-787", "CWE-416"]
RELEVANCE_THRESHOLD       = 0.80
TOP_K_FILTER              = 10
# 每条 BFP 需被至少 CVE_ASSOCIATION_THRESHOLD 个不同的合格 CVE 实例召回才纳入候选
CVE_ASSOCIATION_THRESHOLD = 2   # 关联数量阈值 τ：保留关联到的不同 CVE 数 >= τ 的 BFP
RANDOM_SEED               = 42

# UNSAFE_QUERY_SET_PATH_V2 = PROJECT_ROOT / "data/query/white/unsafe_query_set_v2.json"
# UNSAFE_QUERY_SET_PATH_V2 = PROJECT_ROOT / "data/query/gray/unsafe_query_set_v2.json"
UNSAFE_QUERY_SET_PATH_V2 = PROJECT_ROOT / "data/query/black/unsafe_query_set_v2.json"



def collect_unsafe_bfps_cwe_mixed(
    results: list, target_cwes: list, threshold: float, top_k: int, tau: int
) -> list:
    """从 vul 检索结果中跨 CWE_LIST 汇总含安全隐患的 BFP（统一归入 CWE-MIXED）。

    对于 cwe_id ∈ target_cwes 的每个 CVE 实例，统计其在 rank<=top_k 且 score>=threshold 内
    召回的 BFP；记录每个 BFP 关联到的不同 cve_id 集合，并保留其最高分命中。
    最后仅保留关联 CVE 数 >= tau 的 BFP（按 doc_id 去重）。
    """
    bfp_best = {}                 # doc_id -> best hit
    bfp_cves = defaultdict(set)   # doc_id -> {cve_id, ...}
    target_set = set(target_cwes)
    for entry in results:
        ent = entry.get("entity", {})
        if ent.get("cwe_id") not in target_set:
            continue
        cve_id = (ent.get("cve_id") or "").strip()
        for rank, hit in enumerate(entry.get("retrieval_results", []), start=1):
            if rank > top_k:
                break
            if hit["score"] < threshold:
                continue
            doc_id = hit["doc_id"]
            if cve_id:
                bfp_cves[doc_id].add(cve_id)
            if doc_id not in bfp_best or hit["score"] > bfp_best[doc_id]["score"]:
                bfp_best[doc_id] = hit

    return [
        {"id": h["doc_id"], "language": h.get("language", "C"), "source": h["source"],
         "buggy_code": h["buggy_code"], "fixed_code": h["fixed_code"]}
        for doc_id, h in bfp_best.items()
        if len(bfp_cves[doc_id]) >= tau
    ]


if UNSAFE_QUERY_SET_PATH_V2.exists():
    print(f"[SKIP] 含安全隐患的历史查询集 v2 已存在：{UNSAFE_QUERY_SET_PATH_V2}")
    unsafe_query_set_v2 = load_json(str(UNSAFE_QUERY_SET_PATH_V2))
    print(f"  CWE-MIXED: {len(unsafe_query_set_v2.get('CWE-MIXED', []))} 条")
else:
    # 加载检索结果
    results_s32 = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))

    bfps = collect_unsafe_bfps_cwe_mixed(
        results_s32, CWE_LIST, RELEVANCE_THRESHOLD, TOP_K_FILTER, CVE_ASSOCIATION_THRESHOLD
    )
    unsafe_query_set_v2 = {"CWE-MIXED": bfps}

    UNSAFE_QUERY_SET_PATH_V2.parent.mkdir(parents=True, exist_ok=True)
    save_json(unsafe_query_set_v2, str(UNSAFE_QUERY_SET_PATH_V2))

    print(f"\n===== 含安全隐患的历史查询集 v2 导出汇总 =====")
    print(f"  过滤: score >= {RELEVANCE_THRESHOLD}，rank ∈ [1, {TOP_K_FILTER}]，关联 CVE 数 >= {CVE_ASSOCIATION_THRESHOLD}")
    print(f"  目标 CWE: {CWE_LIST}")
    print(f"  {'-'*42}")
    print(f"  CWE-MIXED  unsafe BFP={len(bfps):>6}")
    print(f"  unsafe → {UNSAFE_QUERY_SET_PATH_V2}")

print("Section 3.2 v2 Done")

2026-06-30 01:27:36 [INFO] src.utils.io: Stream-loaded 17576 records from <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
2026-06-30 01:27:36 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/gray/unsafe_query_set_v2.json



===== 含安全隐患的历史查询集 v2 导出汇总 =====
  过滤: score >= 0.8，rank ∈ [1, 10]，关联 CVE 数 >= 2
  目标 CWE: ['CWE-125', 'CWE-20', 'CWE-119', 'CWE-787', 'CWE-416']
  ------------------------------------------
  CWE-MIXED  unsafe BFP=   752
  unsafe → <PROJECT_ROOT>/data/query/gray/unsafe_query_set_v2.json
Section 3.2 v2 Done


### Section 3.3 v2 — 测试集划分

按 `SPLIT_RATIO` 从含安全隐患的历史查询集 `unsafe_query_set_v2.json` 的 `CWE-MIXED` 单桶中随机划分出测试查询集 `test_query_set_v2.json`（默认取 20%）。剩余 80% 作为候选投毒目标池，在 Section 4 中通过 `unsafe − test` 的集合差现算得到（不单独落盘）。

In [10]:
# ========== Section 3.3 v2: 测试集划分（test_query_set_v2） ==========

SPLIT_RATIO     = 0.8   # 候选投毒目标池 : 测试集 = 8 : 2
RANDOM_SEED_S33 = 42

# TEST_QUERY_SET_PATH_V2 = PROJECT_ROOT / "data/query/white/test/test_query_set_v2.json"
# TEST_QUERY_SET_PATH_V2 = PROJECT_ROOT / "data/query/gray/test/test_query_set_v2.json"
TEST_QUERY_SET_PATH_V2 = PROJECT_ROOT / "data/query/black/test/test_query_set_v2.json"



if TEST_QUERY_SET_PATH_V2.exists():
    print(f"[SKIP] 测试查询集 v2 已存在：{TEST_QUERY_SET_PATH_V2}")
    test_query_set_v2 = load_json(str(TEST_QUERY_SET_PATH_V2))
    print(f"  CWE-MIXED: {len(test_query_set_v2.get('CWE-MIXED', []))} 条")
else:
    # 复用 Section 3.2 的 unsafe_query_set_v2（若内存中不存在则从磁盘加载）
    try:
        unsafe_query_set_v2
    except NameError:
        unsafe_query_set_v2 = load_json(str(UNSAFE_QUERY_SET_PATH_V2))

    bfps = unsafe_query_set_v2.get("CWE-MIXED", [])
    shuffled = bfps[:]
    random.Random(RANDOM_SEED_S33).shuffle(shuffled)
    cut = int(len(shuffled) * SPLIT_RATIO)   # 前 80% 为候选投毒目标，后 20% 为测试集
    test_bfps = shuffled[cut:]
    test_query_set_v2 = {"CWE-MIXED": test_bfps}

    TEST_QUERY_SET_PATH_V2.parent.mkdir(parents=True, exist_ok=True)
    save_json(test_query_set_v2, str(TEST_QUERY_SET_PATH_V2))

    print(f"\n===== 测试集划分汇总 =====")
    print(f"  分割: candidate {SPLIT_RATIO*100:.0f}% / test {100-SPLIT_RATIO*100:.0f}% (seed={RANDOM_SEED_S33})")
    print(f"  {'-'*46}")
    print(f"  {'CWE':<12} {'unsafe':>8} {'candidate':>10} {'test':>7}")
    print(f"  {'-'*46}")
    print(f"  {'CWE-MIXED':<12} {len(bfps):>8} {cut:>10} {len(test_bfps):>7}")
    print(f"  test → {TEST_QUERY_SET_PATH_V2}")

print("Section 3.3 v2 Done")

2026-06-30 01:27:36 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/gray/test/test_query_set_v2.json



===== 测试集划分汇总 =====
  分割: candidate 80% / test 20% (seed=42)
  ----------------------------------------------
  CWE            unsafe  candidate    test
  ----------------------------------------------
  CWE-MIXED         752        601     151
  test → <PROJECT_ROOT>/data/query/gray/test/test_query_set_v2.json
Section 3.3 v2 Done


## Section 4 v2 — 投毒目标选择（CWE-MIXED 单桶聚类 + 预算分配）

对候选投毒目标池（`unsafe_query_set_v2 − test_query_set_v2`，即移除测试集后的 `CWE-MIXED` 单桶）进行 KMeans 聚类（yellowbrick 自动选 k），按投毒率与各簇规模分配预算，从各簇中随机采样投毒目标，产出 `data/query/.../target/poison_targets_set_v2.json`（仅含 `entity` 字段，`retrieval_attack` / `generation_attack` 字段由 Section 5 补全）。

In [11]:
# ========== Section 4 v2: 投毒目标选择（CWE-MIXED 单桶聚类 + 预算分配） ==========
# 产出：data/query/gray/target/poison_targets_set_v2.json

from yellowbrick.cluster import KElbowVisualizer
import matplotlib
matplotlib.use("Agg")  # 关闭 GUI 后端，避免 notebook 中 yellowbrick 的渲染开销

K_RANGE_V2        = (2, 15)        # yellowbrick KElbowVisualizer 搜索范围
DEFAULT_K_V2      = 5              # 无法识别肘点时的回退簇数
POISON_RATE_V2    = 0.0065         # 总投毒率θ（方案一默认 0.75%）
APR_CORPUS_SIZE   = 50000          # 与 Section 1.2 的 CORPUS_SIZE 一致
RANDOM_SEED_V2    = 42

# POISON_TARGETS_OUT_V2 = PROJECT_ROOT / "data/query/white/target/poison_targets_set_v2.json"
# POISON_TARGETS_OUT_V2 = PROJECT_ROOT / "data/query/gray/target/poison_targets_set_v2.json"
POISON_TARGETS_OUT_V2 = PROJECT_ROOT / "data/query/black/target/poison_targets_set_v2.json"



if POISON_TARGETS_OUT_V2.exists():
    print(f"[SKIP] 投毒目标文件 v2 已存在：{POISON_TARGETS_OUT_V2}")
    poison_targets_v2 = load_json(str(POISON_TARGETS_OUT_V2))
    print(f"  CWE-MIXED: {len(poison_targets_v2.get('CWE-MIXED', []))} 条")
else:
    # 构建候选投毒目标池：unsafe_query_set_v2 − test_query_set_v2（按 id 移除测试集元素）
    unsafe_data_v2 = load_json(str(UNSAFE_QUERY_SET_PATH_V2))
    test_data_v2   = load_json(str(TEST_QUERY_SET_PATH_V2))
    test_ids = {item["id"] for item in test_data_v2.get("CWE-MIXED", [])}
    subset = [
        item for item in unsafe_data_v2.get("CWE-MIXED", []) if item["id"] not in test_ids
    ]
    if not subset:
        raise ValueError("CWE-MIXED 候选池为空，无法进行投毒目标选择")

    # 加载嵌入模型（与 BFP-APR 库一致，便于语义对齐）
    apr_db_config_s4 = load_yaml(str(PROJECT_ROOT / "configs/database/bfp/apr.yml"))
    embedder_v2 = load_embedder_from_db_config(apr_db_config_s4) 
    

    # ── 步骤1: 计算总投毒预算 ──
    total_budget_v2 = round(POISON_RATE_V2 * APR_CORPUS_SIZE)
    print(f"\n总投毒预算: {total_budget_v2} (POISON_RATE × APR_CORPUS = {POISON_RATE_V2}×{APR_CORPUS_SIZE})")
    print(f"CWE-MIXED 子集规模: {len(subset)}")

    # ── 步骤2: 嵌入 + yellowbrick KElbow 自动选 k ──
    print(f"\n{'='*60}")
    print(f"[CWE-MIXED] 子集规模 {len(subset)}，预算 {total_budget_v2}")

    codes_mixed = [item["buggy_code"] for item in subset]
    emb_mixed   = np.asarray(embedder_v2.embed_documents(codes_mixed))

    k_max_eff = max(2, min(K_RANGE_V2[1], len(subset) - 1))
    try:
        visualizer = KElbowVisualizer(
            KMeans(n_init='auto', random_state=RANDOM_SEED_V2),
            k=(K_RANGE_V2[0], k_max_eff + 1),
            timings=False,
            force_model=True
        )
        visualizer.fit(emb_mixed)
        best_k = int(visualizer.elbow_value_) if visualizer.elbow_value_ else DEFAULT_K_V2
    except Exception as e:
        print(f"  yellowbrick 选 k 失败 ({e})，回退 DEFAULT_K_V2={DEFAULT_K_V2}")
        best_k = DEFAULT_K_V2
    best_k = max(1, min(best_k, len(subset)))
    print(f"  自动选定 k = {best_k}")

    # ── 步骤3: KMeans 聚类 ──
    kmeans_mixed = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_SEED_V2)
    cluster_labels = kmeans_mixed.fit_predict(emb_mixed)

    # 按簇分组
    clusters = defaultdict(list)
    for idx, label in enumerate(cluster_labels):
        clusters[int(label)].append(subset[idx])

    # ── 步骤4: 簇内预算分配（按簇规模加权）+ 簇内随机抽样 ──
    rng_v2 = random.Random(RANDOM_SEED_V2)
    cluster_sizes = {c: len(items) for c, items in clusters.items()}
    subset_total_size = sum(cluster_sizes.values())
    targets_mixed = []
    used_total = 0
    for cluster_id, items in clusters.items():
        cluster_quota = round(total_budget_v2 * cluster_sizes[cluster_id] / subset_total_size)
        cluster_quota = min(cluster_quota, len(items))
        if cluster_quota <= 0:
            continue
        sampled = rng_v2.sample(items, cluster_quota)
        for s in sampled:
            targets_mixed.append({
                "entity": {
                    "id"        : s["id"],
                    "language"  : s.get("language", "C"),
                    "source"    : s["source"],
                    "buggy_code": s["buggy_code"],
                    "fixed_code": s["fixed_code"],
                }
            })
        used_total += cluster_quota
        print(f"  簇 {cluster_id}: 大小 {cluster_sizes[cluster_id]:>4d}，配额 {cluster_quota:>3d}")

    poison_targets_v2 = {"CWE-MIXED": targets_mixed}
    print(f"\n[CWE-MIXED] 总投毒目标: {len(targets_mixed)} 条（预算 {total_budget_v2}，实际命中 {used_total}）")

    # 释放嵌入模型
    release_embedder(embedder_v2)

    # ── 步骤5: 写出 ──
    POISON_TARGETS_OUT_V2.parent.mkdir(parents=True, exist_ok=True)
    save_json(poison_targets_v2, str(POISON_TARGETS_OUT_V2))
    print(f"\n✅ 投毒目标文件 v2 已保存: {POISON_TARGETS_OUT_V2}")

    print(f"\n===== 汇总 =====")
    print(f"  {'CWE':<12} {'代理查询数':>10} {'预算':>6} {'命中':>6}")
    print(f"  {'-'*38}")
    print(f"  {'CWE-MIXED':<12} {len(subset):>10} {total_budget_v2:>6} {len(targets_mixed):>6}")

print("\nSection 4 v2 Done — 投毒目标采样完成 ✅")

2026-06-30 01:30:23 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/gray/unsafe_query_set_v2.json
2026-06-30 01:30:23 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/gray/test/test_query_set_v2.json
2026-06-30 01:30:23 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-06-30 01:30:23 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-06-30 01:30:23 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-06-30 01:30:23 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-06-30 01:30:23 [INFO] src.utils.device: 预筛选后的设备池: ['cuda:0']
2026-06-30 01:30:23 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8187 MiB)
2026-06-30 01:30:23 [INFO] src.models.retriever.GTE: Loading model: Alibaba-NLP/gte-modernbert-base on cuda:0
2026-06-30 01:30:23 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: Alibaba-NLP/gte-modernbert-base
`torch_dtype` is depre


总投毒预算: 375 (POISON_RATE × APR_CORPUS = 0.0075×50000)
CWE-MIXED 子集规模: 601

[CWE-MIXED] 子集规模 601，预算 375


Batches: 100%|██████████| 601/601 [00:43<00:00, 13.75it/s]


  自动选定 k = 8
  簇 4: 大小   91，配额  57
  簇 3: 大小   58，配额  36
  簇 5: 大小  138，配额  86
  簇 2: 大小   40，配额  25
  簇 6: 大小   86，配额  54
  簇 1: 大小  117，配额  73
  簇 0: 大小   37，配额  23
  簇 7: 大小   34，配额  21

[CWE-MIXED] 总投毒目标: 375 条（预算 375，实际命中 375）


2026-06-30 01:31:24 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v2.json


[GPU] 嵌入模型已释放

✅ 投毒目标文件 v2 已保存: <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v2.json

===== 汇总 =====
  CWE               代理查询数     预算     命中
  --------------------------------------
  CWE-MIXED           601    375    375

Section 4 v2 Done — 投毒目标采样完成 ✅


## Section 5 v2 — 正样本选择（Vinj 检索 + 加权采样）

对 Section 4 v2 产出的每个投毒目标（`CWE-MIXED` 单桶）：
1. 使用 `entity.fixed_code` 在 Vul-Vinj 库中跨 `CWE_LIST` 检索相关 CVE 实例（Top-`TOP_K_VINJ`），再以**不放回加权抽样**选出 `RELEVANT_VUL_NUM` 条作为 `generation_attack.vinj_attack.relevant_vul`：初始权重为相关性分数（分数越高越易抽中），每抽中一个 `cve_id` 后将剩余同 `cve_id` 候选的权重乘以衰减因子 `RELEVANT_VUL_DECAY_GAMMA`（提升 `cve_id` 多样性）
2. 从 Section 3.1 的 BFP↔CVE 检索结果中，反查这些 CVE 被哪些 BFP 召回（最多 `RELEVANT_VUL_NUM × TOP_K` 条候选）
3. 按召回分数加权采样 `POSITIVE_SAMPLE_NUM` 条 BFP 作为 `retrieval_attack.positive_samples`
4. 回写覆盖 `poison_targets_set_v2.json`

In [12]:
# ========== Section 5 v2: 正样本选择（Vinj 检索 + 加权采样） ==========

RELEVANT_VUL_NUM_V2      = 5
POSITIVE_SAMPLE_NUM_V2   = 9
TOP_K_VINJ_V2            = 10
RANDOM_SEED_V2_S5        = 42
RELEVANT_VUL_DECAY_GAMMA = 0.9   # 同 cve_id 多样性衰减因子 γ ∈ (0,1)


def weighted_sample_relevant_vul(hits, num, gamma, rng):
    """不放回加权抽样 relevant_vul（序贯 + 同 cve_id 多样性衰减）。

    初始权重 = 相关性分数（score）；每抽中一个候选后，将剩余“同 cve_id”
    候选的权重乘以衰减因子 gamma，从而实现“抽到相同 cve_id 越多，
    同属该 cve_id 但未被抽取的候选权重越低”的多样性约束。
    """
    pool     = list(hits)
    weights  = [max(float(h.get("score", 0.0)), 1e-6) for h in pool]
    cves     = [(h.get("cve_id") or "").strip() for h in pool]
    selected = []
    while pool and len(selected) < num:
        total = sum(weights)
        if total <= 0:
            break
        probs = np.array(weights, dtype=np.float64) / total
        idx = int(np.random.default_rng(rng.randint(0, 2**31 - 1)).choice(len(pool), p=probs))
        chosen_cve = cves[idx]
        selected.append(pool.pop(idx))
        weights.pop(idx)
        cves.pop(idx)
        for j, c in enumerate(cves):       # 同 cve_id 衰减
            if c and c == chosen_cve:
                weights[j] *= gamma
    return selected


# ── 字段级幂等检查：首条记录已含 relevant_vul / positive_samples 即视为已完成 ──
def _v2_is_already_enriched(targets_dict) -> bool:
    items = targets_dict.get("CWE-MIXED", [])
    if not items:
        return False
    head = items[0]
    has_pos = bool(head.get("retrieval_attack", {}).get("positive_samples"))
    has_vul = bool(head.get("generation_attack", {}).get("vinj_attack", {}).get("relevant_vul"))
    return has_pos and has_vul

poison_targets_v2 = load_json(str(POISON_TARGETS_OUT_V2))

if _v2_is_already_enriched(poison_targets_v2):
    print(f"[SKIP] 投毒目标文件已含 relevant_vul / positive_samples 字段：{POISON_TARGETS_OUT_V2}")
else:
    # ── 步骤1: 构建 cve_id → {doc_id: best_hit} 倒排索引 ──
    print("构建 cve_id → BFP 倒排索引...")
    proxy_retrieval_results_v2 = load_json_stream(str(PROXY_RETRIEVAL_RESULTS_PATH))

    cve_to_bfps = defaultdict(dict)  # cve_id -> {bfp_doc_id: best_hit}
    for entry in proxy_retrieval_results_v2:
        ent = entry.get("entity", {})
        cve_id = (ent.get("cve_id") or "").strip()
        if not cve_id:
            continue
        for hit in entry.get("retrieval_results", []):
            doc_id = hit.get("doc_id")
            if doc_id is None:
                continue
            score = float(hit.get("score", 0.0))
            existing = cve_to_bfps[cve_id].get(doc_id)
            if existing is None or score > float(existing.get("score", 0.0)):
                cve_to_bfps[cve_id][doc_id] = hit
    print(f"  倒排索引覆盖 {len(cve_to_bfps)} 个 cve_id")

    # ── 步骤2: 加载 Vul-Vinj 库 + embedder ──
    vinj_db_config_s5 = load_yaml(str(PROJECT_ROOT / "configs/database/vul/vinj.yml"))
    if not os.path.isabs(vinj_db_config_s5["database"]["uri"]):
        vinj_db_config_s5["database"]["uri"] = str(PROJECT_ROOT / vinj_db_config_s5["database"]["uri"])

    vinj_milvus_s5    = VulMilvusClient(vinj_db_config_s5)
    embedder_v2_s5    = load_embedder_from_db_config(vinj_db_config_s5)
    vinj_retriever_s5 = VulRetriever(embedder_v2_s5, vinj_milvus_s5)

    targets = poison_targets_v2.get("CWE-MIXED", [])
    print(f"\n[CWE-MIXED] 投毒目标数: {len(targets)}")

    # ── 步骤3: 批量检索：以 entity.fixed_code 为查询，跨 CWE_LIST 检索相关 CVE ──
    queries_s5 = [
        {
            "id"        : t["entity"]["id"],
            "language"  : "C",
            "source"    : t["entity"].get("source", ""),
            "buggy_code": t["entity"].get("buggy_code", ""),
            "fixed_code": t["entity"]["fixed_code"],
        }
        for t in targets
    ]
    retrieved = vinj_retriever_s5.dense_retrieve(
        queries_s5,
        top_k=TOP_K_VINJ_V2,
        query_target="fixed_code",
        cwe_list=CWE_LIST,
    )

    # 构建 entity_id → retrieval_results 映射
    ret_map = {r["id"]: r.get("retrieval_results", []) for r in retrieved}

    # ── 步骤4: 逐目标处理：保留 Top-RELEVANT_VUL_NUM 个 CVE + 加权采样正样本 ──
    rng_v2_s5 = random.Random(RANDOM_SEED_V2_S5)
    n_with_vul = 0
    n_with_pos = 0
    for tgt in targets:
        ent_id = tgt["entity"]["id"]
        hits = ret_map.get(ent_id, [])

        # 仅保留命中 CWE_LIST 内类型的实例（dense_retrieve 已用 cwe_list 过滤，此处防御性检查）
        filtered_hits = [h for h in hits if h.get("cwe_id", "") in CWE_LIST]
        # 不放回加权抽样：按 score 加权，并对同 cve_id 施加序贯多样性衰减
        cwe_hits = weighted_sample_relevant_vul(
            filtered_hits, RELEVANT_VUL_NUM_V2, RELEVANT_VUL_DECAY_GAMMA, rng_v2_s5
        )

        # 构造 relevant_vul 列表
        relevant_vul = []
        for h in cwe_hits:
            relevant_vul.append({
                "doc_id"     : h.get("doc_id"),
                "chunk_id"   : h.get("chunk_id", 0),
                "cluster_id" : h.get("cluster_id", ""),
                "source"     : h.get("source", ""),
                "total_chunks": h.get("total_chunks", 1),
                "vul_code"   : h.get("vul_code", ""),
                "cwe_id"     : h.get("cwe_id", ""),
                "cve_id"     : h.get("cve_id", ""),
                "cwe_desc"   : h.get("cwe_desc", ""),
                "cve_desc"   : h.get("cve_desc", ""),
                "vul_pattern": "",
                "score"      : h.get("score", 0.0),
            })

        if relevant_vul:
            n_with_vul += 1

        # 加权采样正样本：从这些 CVE 召回过的 BFP 中按分数采样
        candidate_bfps = {}  # doc_id -> (score, bfp)
        for vul in relevant_vul:
            cve_id = (vul.get("cve_id") or "").strip()
            if not cve_id:
                continue
            for doc_id, bfp in cve_to_bfps.get(cve_id, {}).items():
                if doc_id == ent_id:  # 排除目标自身
                    continue
                score_b = float(bfp.get("score", 0.0))
                if doc_id not in candidate_bfps or score_b > candidate_bfps[doc_id][0]:
                    candidate_bfps[doc_id] = (score_b, bfp)

        positive_samples = []
        if candidate_bfps:
            items_list = list(candidate_bfps.values())
            weights    = [max(s, 1e-6) for s, _ in items_list]
            k_eff      = min(POSITIVE_SAMPLE_NUM_V2, len(items_list))
            if k_eff < POSITIVE_SAMPLE_NUM_V2:
                print(f"  [CWE-MIXED] 目标 id={ent_id} 候选 BFP 仅 {k_eff} 条 (< {POSITIVE_SAMPLE_NUM_V2})")
            probs = np.array(weights, dtype=np.float64)
            probs /= probs.sum()
            idx_sel = np.random.default_rng(rng_v2_s5.randint(0, 2**31 - 1)).choice(
                len(items_list), size=k_eff, replace=False, p=probs
            )
            for i in idx_sel:
                _, bfp = items_list[int(i)]
                positive_samples.append({
                    "id"        : bfp.get("doc_id"),
                    "source"    : bfp.get("source", ""),
                    "buggy_code": bfp.get("buggy_code", ""),
                    "fixed_code": bfp.get("fixed_code", ""),
                })

        if positive_samples:
            n_with_pos += 1

        tgt["retrieval_attack"] = {"positive_samples": positive_samples}
        tgt["generation_attack"] = {
            "vinj_attack"     : {"relevant_vul": relevant_vul},
            "jailbreak_attack": {},
        }

    # 关闭 Milvus + 释放 embedder
    vinj_milvus_s5.close()
    release_embedder(embedder_v2_s5)

    # ── 步骤5: 回写覆盖 ──
    save_json(poison_targets_v2, str(POISON_TARGETS_OUT_V2))
    print(f"\n[CWE-MIXED] 含 relevant_vul: {n_with_vul}/{len(targets)}，含 positive_samples: {n_with_pos}/{len(targets)}")
    print(f"✅ 投毒目标文件（含 positive_samples & relevant_vul）已回写: {POISON_TARGETS_OUT_V2}")

print("\nSection 5 v2 Done — 正样本选择完成 ✅")

2026-06-30 01:31:24 [INFO] src.utils.io: Loaded JSON from <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v2.json


构建 cve_id → BFP 倒排索引...


2026-06-30 01:31:49 [INFO] src.utils.io: Stream-loaded 17576 records from <PROJECT_ROOT>/data/query/gray/num_17576_alpha_1_target_vul_code_retriever_harrier-oss-v1-0.6b_retrieval_results.json
2026-06-30 01:31:49 [INFO] src.rag.vul.milvus_client: 从模型配置 <PROJECT_ROOT>/configs/models/harrier-oss-v1-0.6b.yml 读取 dim=1024
2026-06-30 01:31:49 [INFO] src.rag.vul.milvus_client: Connecting to Milvus at <PROJECT_ROOT>/data/milvus/vul/harrier-oss-v1-0.6b.db
2026-06-30 01:31:49 [INFO] src.rag.vul.milvus_client: Milvus client connected
2026-06-30 01:31:49 [INFO] src.utils.device: nvidia-smi 探测完成: 健康 GPU = [0]
2026-06-30 01:31:49 [WARNING] src.utils.device: 设备池预筛选排除 cuda:1（nvidia-smi 判定为不健康）
2026-06-30 01:31:49 [WARNING] src.utils.device: 设备池预筛选排除 cuda:2（nvidia-smi 判定为不健康）
2026-06-30 01:31:49 [WARNING] src.utils.device: 设备池预筛选排除 cuda:3（nvidia-smi 判定为不健康）
2026-06-30 01:31:49 [INFO] src.utils.device: 预筛选后的设备池: ['cuda:0']
2026-06-30 01:31:49 [INFO] src.utils.device: 设备池探测成功: cuda:0 (NVIDIA GeForce RTX 4

  倒排索引覆盖 4912 个 cve_id


2026-06-30 01:31:49 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: microsoft/harrier-oss-v1-0.6b
2026-06-30 01:32:00 [INFO] sentence_transformers.SentenceTransformer: 3 prompts are loaded, with the keys: ['web_search_query', 'sts_query', 'bitext_query']
2026-06-30 01:32:00 [INFO] src.models.retriever.Harrier: Model loaded. dim=1024, task=code2code



[CWE-MIXED] 投毒目标数: 375


Vul dense batch retrieval: 100%|██████████| 2/2 [00:27<00:00, 13.51s/it]
2026-06-30 01:33:13 [INFO] src.rag.vul.retriever: Dense batch retrieval completed: 375 queries, top_k=10, cluster_search=False, shards=2
2026-06-30 01:33:13 [INFO] src.rag.vul.milvus_client: Milvus client closed


[GPU] 嵌入模型已释放


2026-06-30 01:33:17 [INFO] src.utils.io: Saved results to <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v2.json



[CWE-MIXED] 含 relevant_vul: 375/375，含 positive_samples: 375/375
✅ 投毒目标文件（含 positive_samples & relevant_vul）已回写: <PROJECT_ROOT>/data/query/gray/target/poison_targets_set_v2.json

Section 5 v2 Done — 正样本选择完成 ✅
